# Phase 4, Stage 1: Descriptive Statistics

This notebook computes the baseline descriptive picture for each of the 20 cells (5 rating bands x 4 time pressure bins) before any formal hypothesis testing.

For each cell, on `capped_cpl`, we compute:
- n (sample size)
- mean, median, standard deviation, IQR
- skewness, kurtosis (shape descriptors)
- proportion of moves in each error category (Inaccuracy / Minor Error / Major Error / Blunder)

See `../CLAUDE.md` for full project context, definitions, and the overall analysis plan.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Label mappings, taken from ../../config.py
RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

TIME_PRESSURE_LABELS = {
    1: 'Minimal (>75%)',
    2: 'Low (50-75%)',
    3: 'Moderate (25-50%)',
    4: 'High (<25%)',
}

ERROR_CATEGORY_LABELS = {
    1: 'Inaccuracy',
    2: 'Minor Error',
    3: 'Major Error',
    4: 'Blunder',
}

In [ ]:
# Load the Phase 3 analytical dataset
df = pd.read_csv('../../data/processed/analysed_moves.csv')

df['rating_band_label'] = df['rating_band'].map(RATING_BAND_LABELS)
df['time_pressure_label'] = df['time_pressure_bin'].map(TIME_PRESSURE_LABELS)

print(f'Loaded {len(df):,} rows')
df.head()

## Computing the descriptive statistics

A quick refresher on the two shape statistics, since these are central to the distributional framing of this project:

- **Skewness** measures asymmetry. A skewness of 0 means the distribution is symmetric. `capped_cpl` is bounded at 0 (no negative CPL) and capped at 300, so we expect strong **positive skew** in every cell (most moves are near-perfect, with a long right tail of errors). The question is whether that skew gets *more* extreme under time pressure.
- **Kurtosis** (we use *excess* kurtosis, where 0 = same tail-heaviness as a normal distribution) measures how much of the data sits in the tails vs the centre. Higher kurtosis = more extreme values (more blunders) relative to the bulk of small errors.

We use `scipy.stats.skew`/`kurtosis` with `bias=False` (the sample-corrected versions), since these are the standard formulas reported in most statistics software.

In [ ]:
grouped = df.groupby(['rating_band', 'time_pressure_bin'])['capped_cpl']

# Core location/spread/shape statistics
desc = grouped.agg(
    n='count',
    mean='mean',
    median='median',
    std='std',
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75),
    skewness=lambda x: stats.skew(x, bias=False),
    kurtosis=lambda x: stats.kurtosis(x, fisher=True, bias=False),
).reset_index()

desc['iqr'] = desc['q3'] - desc['q1']
desc = desc.drop(columns=['q1', 'q3'])

desc.head()

## Error category proportions

For each cell, what fraction of moves fall into each error category? This is the categorical complement to the continuous shape statistics above, and feeds directly into the Stage 3 chi-squared tests and the Stage 5 stacked bar charts.

In [ ]:
# Count moves per (rating_band, time_pressure_bin, error_category), then convert to
# proportions within each (rating_band, time_pressure_bin) cell.
counts = df.groupby(['rating_band', 'time_pressure_bin', 'error_category']).size()
proportions = counts / counts.groupby(level=[0, 1]).sum()

error_props = proportions.unstack('error_category')
error_props.columns = [f'prop_{ERROR_CATEGORY_LABELS[c]}' for c in error_props.columns]
error_props = error_props.reset_index()

error_props.head()

## Combine into one summary table

Merge the shape statistics and the error category proportions into a single table, one row per cell, and save it to `../results/` so we can reuse it in later notebooks (and pull straight from it when writing up the Results section).

In [ ]:
summary = desc.merge(error_props, on=['rating_band', 'time_pressure_bin'])

summary['rating_band_label'] = summary['rating_band'].map(RATING_BAND_LABELS)
summary['time_pressure_label'] = summary['time_pressure_bin'].map(TIME_PRESSURE_LABELS)

# Reorder columns for readability
cols = ['rating_band', 'rating_band_label', 'time_pressure_bin', 'time_pressure_label',
        'n', 'mean', 'median', 'std', 'iqr', 'skewness', 'kurtosis',
        'prop_Inaccuracy', 'prop_Minor Error', 'prop_Major Error', 'prop_Blunder']
summary = summary[cols].sort_values(['rating_band', 'time_pressure_bin']).reset_index(drop=True)

summary.to_csv('../results/descriptive_stats.csv', index=False)
print('Saved to ../results/descriptive_stats.csv')
summary